In [1]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel,Field
import time

from langchain_community.document_loaders import PyPDFLoader,JSONLoader
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

g:\Genarative-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from llama_index.core.vector_stores import MetadataInfo, VectorStoreInfo
from llama_index.core.retrievers import VectorIndexAutoRetriever

In [ ]:
def matadata_making_funtion(record: dict, metadata:dict) -> dict:
    metadata['name'] = record.get('name')
    metadata['doc_type'] = 'teacher_details'
    metadata['position'] = record.get('designation')
    metadata['department'] = record.get('department').lower()
    metadata['phone'] = record.get('phone')
    
    

    return metadata

docs = JSONLoader(
        file_path='.\\Document\\teachers_data.json',

        jq_schema='.[]', 
        content_key= 'context_text',
        metadata_func= matadata_making_funtion
    ).load()

print(len(docs))
print(docs[2])

87
page_content='দূর্গা চরন রায় হলেন (চাঁপাইনবাবগঞ্জ পলিটেকনিক ইনস্টিটিউট / CNPI)-এর একজন চীফ ইন্সট্রাক্টর / CI। তার পদবি হলো চীফ ইন্সট্রাক্টর / CI ( নন-টেক) রসায়ন। তিনি নন-টেক বিভাগে কর্মরত আছেন। তার সাথে যোগাযোগ করতে পারবেন এই নম্বরে: ০১৭১৫৫৮৭৩৪৫। দূর্গা চরন রায়-এর ফোন নম্বর ০১৭১৫৫৮৭৩৪৫। যদি কেউ নন-টেক বিভাগের শিক্ষকদের তথ্য জানতে চান, তাহলে দূর্গা চরন রায় (চীফ ইন্সট্রাক্টর / CI ( নন-টেক) রসায়ন) একজন গুরুত্বপূর্ণ যোগাযোগ ব্যক্তি।' metadata={'source': 'G:\\Genarative-AI\\Self Rag\\Document\\teachers_data.json', 'seq_num': 3, 'name': 'দূর্গা চরন রায়', 'user_type': 'teacher', 'position': 'চীফ ইন্সট্রাক্টর ( নন-টেক) রসায়ন', 'department': 'non-tech', 'phone': '০১৭১৫৫৮৭৩৪৫'}


In [4]:
chunks = RecursiveCharacterTextSplitter(chunk_size = 600,chunk_overlap = 150).split_documents(docs)
print(len(chunks))

87


In [5]:
embed_model = HuggingFaceEmbeddings(model="BAAI/bge-m3")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5959.79it/s]


In [6]:
import os
# llm = ChatGroq(
#     model="groq/compound",
#     api_key=os.getenv("GROQ_API_KEY"),
#     model_kwargs=  {'response_formate': {'type':'json_object'}}
# )

llm = ChatGroq(
    temperature=0,
    model="meta-llama/llama-4-scout-17b-16e-instruct", 
   
)

In [7]:
import lark
vector_store = Chroma.from_documents(docs,embed_model)

# retriever = SelfQueryRetriever.from_llm(
#     llm=llm, 
#     vectorstore=vector_store, 
#     document_contents=document_content_description, 
#     metadata_field_info=metadata_field_info, 
#     enable_limit=True, 
#     verbose=True 
# )

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [8]:
from typing import Annotated, TypedDict
class State(TypedDict):
    question:str

    retrieval_query: str
    rewrite_tries: int

    need_retrieval:bool

    docs:list[Document]
    relevant_docs:list[Document]
    context:str

    issup:Literal["fully_supported","partially_supported","no_support"]
    evidence:list[str]

    isuse: Literal['useful','not_useful']
    use_reason: str
    
    retries: int
    out:list
    answer: Annotated[str, lambda old, new: new]

In [9]:
from langchain_core.output_parsers import StrOutputParser
class RetrieveDecision(BaseModel):
    should_retrieve:bool = Field(
        ...,
        description = "True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}")
    ]
)

retrieve_decied_chain = decide_retrieval_prompt | llm.with_structured_output(
    RetrieveDecision,
    method="json_mode"  
)
def decide_retrieval_node(state: State):
    decision:RetrieveDecision = retrieve_decied_chain.invoke({'question':state['question']})

    return {'need_retrieval':decision.should_retrieve}

In [10]:
class Isrelevant(BaseModel):
    is_relevant:bool = Field(
        ...,
        description='True if the document helps answer the question, else False.'
    )

Isrelevant_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',
        "You are judging document relevance.\n"
        "Return JSON that matches this schema:\n"
        "{{'is_relevant': boolean}}\n\n"
        "2. ROLE/ENTITY CHECK: If the question asks for a specific role or person (e.g., 'Principal' or 'অধ্যক্ষ') and the document describes a different role (e.g., 'Vice Principal' or 'ইন্সট্রাক্টর'), the document is NOT relevant, even if they belong to the same institute. Return {{'is_relevant': false}}.\n"
        ),
        ('human','\nQuestion: {question}\n\nDocument:\n{document}')
    ]
)


isrelevant_chain = Isrelevant_prompt | llm.with_structured_output(Isrelevant,method='json_mode') 

def isrelevant_node(state:State):
    docs = state['docs']
    qus = state['question']
    all_docs = []
    all_docs_input = []
    relevanced_doc:list[Document] = []

    for d  in docs:
       
        all_docs_input.append({'question':qus,'document':d.page_content})
    
    all_docs_out = isrelevant_chain.batch(all_docs_input)

    for d,r in zip(docs,all_docs_out):
        if r.is_relevant:
            relevanced_doc.append(d)

    return {
        'relevant_docs': relevanced_doc,
        'out':all_docs_out
        
        }
        


In [11]:
generation_prompt = ChatPromptTemplate.from_messages([
     (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", '\nQuestion: {question}')]
        
)

genaration_chain = generation_prompt | llm | StrOutputParser()

def generation_node(state:State):
    out = genaration_chain.invoke({'question':state['question']})
    return {'answer': out, 'context': ''}

In [12]:
main_generation_prompt = ChatPromptTemplate.from_messages(
    [
         ('system',
        "You are a strict business RAG assistant.\n"
        "Answer the user's question using ONLY the provided context.\n"
        "RULES:\n"
        "1. Compare the entities (like company names) in the Question and the Context. If the Question asks about a specific entity (e.g., DiplomaAI) but the Context is about a different entity (e.g., NexaAI), you MUST say: 'No relevant document found.'\n"
        "2. When generating your answer, ALWAYS explicitly mention the specific entity/company name from the context (e.g., say 'Aarav Mehta is the CEO of NexaAI' instead of just 'Aarav Mehta is the CEO').\n"
        "3. Do not use outside knowledge.\n"
        "RULES:\n"
        "1. Answer ONLY what the user explicitly asked for. Provide a direct, short, and concise answer.\n"
        "2. DO NOT include any extra or additional information from the context that was not requested (e.g., if asked about the Principal, do NOT mention the Vice-Principal or Instructors).\n"
        "3. Compare the entities in the Question and Context. If not matching, say: 'No relevant document found.'\n"
        "4. Do not use outside knowledge.\n"
        ),   
        ('human','\nquestion:{question}\n\ndocument:{context}')
    ]
)

main_generation_chain = main_generation_prompt | llm | StrOutputParser()

def main_generation_node(state:State):
    qus = state['question']
    docs = state.get('relevant_docs',[])

    context = "\n\n---\n\n".join(d.page_content for d in docs).strip()
    if not context:
        return{'answer':"I dont Know,No relevant document found", 'context': ''}
        
    ans = main_generation_chain.invoke({'question':qus,'context':context})
    return {
        'answer':ans,
        'context':context
        }

In [13]:
def no_relevant_docs(state:State):
    return{'answer':"I dont Know,No relevant document found", 'context': ''}


In [14]:
class IsSupDecision(BaseModel):
    issup: Literal["fully_supported","partially_supported","no_support"]
    evidence: list[str] = Field(default_factory=list)

issup_prompt = ChatPromptTemplate([
    ('system',
    "You are verifying whether the ANSWER is supported by the CONTEXT.\n"
            "Return JSON with keys: issup, evidence.\n"
            "issup must be one of: fully_supported, partially_supported, no_support.\n\n"
            "How to decide issup:\n"
            "- fully_supported:\n"
            "  Every meaningful claim is explicitly supported by CONTEXT, and the ANSWER does NOT introduce\n"
            "  any qualitative/interpretive words that are not present in CONTEXT.\n"
            "  (Examples of disallowed words unless present in CONTEXT: culture, generous, robust, designed to,\n"
            "  supports professional development, best-in-class, employee-first, etc.)\n\n"
            "- partially_supported:\n"
            "  The core facts are supported, BUT the ANSWER includes ANY abstraction, interpretation, or qualitative\n"
            "  phrasing not explicitly stated in CONTEXT (e.g., calling policies 'culture', saying leave is 'generous',\n"
            "  or inferring outcomes like 'supports professional development').\n\n"
            "- no_support:\n"
            "  The key claims are not supported by CONTEXT.\n\n"
            "Rules:\n"
            "- Be strict: if you see ANY unsupported qualitative/interpretive phrasing, choose partially_supported.\n"
            "- If the answer is mostly unrelated to the question or unsupported, choose no_support.\n"
            "- Evidence: list up to 3 short direct quotes from CONTEXT as plain strings only. Each evidence item MUST be a string, not an object or dict.\n"
            "- Do not use outside knowledge."
            "- Entity check: If the ANSWER contains a company name, product name, or proper noun "
            "that does NOT appear in CONTEXT, classify as no_support immediately.\n"
            "- Entity Check: Compare the entities in the ANSWER and the CONTEXT. If the ANSWER assigns properties to an entity from the QUESTION that does not exist in the CONTEXT, or if the ANSWER hides the specific company name to appear correct, choose no_support.\n"
            ),
    (
            "human",
            "Question:\n{question}\n\n"
            "Answer:\n{answer}\n\n"
            "Context:\n{context}\n"
        )
])

issup_chain = issup_prompt | llm.with_structured_output(IsSupDecision,method='json_mode')

def issup_node(state:State):
    qus =  state['question']
    answer = state['answer']
    context = state.get('context', '')  # safe get

    if not context:
        return {
            'issup': 'no_support',
            'evidence': []
        }

    decision:IsSupDecision = issup_chain.invoke({'question':qus,'answer':answer,'context':context})
    return {
        'issup':decision.issup,
        'evidence':decision.evidence
    }

In [15]:
revise_prompt = ChatPromptTemplate([
    (
            "system",
            "You are a STRICT reviser.\n\n"
            "You must output based on the following format:\n\n"
            "FORMAT (quote-only answer):\n"
            "- <direct quote from the CONTEXT>\n"
            "- <direct quote from the CONTEXT>\n\n"
            "Rules:\n"
            "- Use ONLY the CONTEXT.\n"
            "- Do NOT add any new words besides bullet dashes and the quotes themselves.\n"
            "- Do NOT explain anything.\n"
            "- Do NOT say 'context', 'not mentioned', 'does not mention', 'not provided', etc.\n"
        ),
        (
            'human',
            'Question:\n{question}\n\ncurrent answer:\n{answer}\n\ncontext:\n{context}'
        )
])

revise_chain = revise_prompt | llm | StrOutputParser()

def revise_node(state:State):
    current_ans = state['answer']
    qus = state['question']
    context = state.get('context', '')

    out = revise_chain.invoke({'question':qus,'answer':current_ans,'context':context})

    return {
        'answer':out,
        'retries':state.get('retries',0) + 1 
    }

In [16]:
class IsUse(BaseModel):
    isuse: Literal['useful','not_useful'] 
    use_reason: str = Field(...,description='Short reason in 1 line')

isuse_prompt = ChatPromptTemplate.from_messages([
    (
            "system",
            "You are judging USEFULNESS of the ANSWER for the QUESTION.\n\n"
            "Goal:\n"
            "- Decide if the answer actually addresses what the user asked.\n\n"
            "Return JSON with keys: isuse, reason.\n"
            "isuse must be one of: useful, not_useful.\n\n"
            "Rules:\n"
            "- useful: The answer directly answers the question or provides the requested specific info.\n"
            "- not_useful: The answer is generic, off-topic, or only gives related background without answering.\n"
            "- Do NOT use outside knowledge.\n"
            "- Do NOT re-check grounding (IsSUP already did that). Only check: 'Did we answer the question?'\n"
            "- Keep reason to 1 short line."
            "- not_useful: The answer is generic, off-topic, or talks about a DIFFERENT entity/company than what the user explicitly asked for in the question.\n"
        ),
        (
            "human",
            "Question:\n{question}\n\nAnswer:\n{answer}"
        )
])

isuse_chain = isuse_prompt | llm.with_structured_output(IsUse)

def isuse_node(state:State):
    qus = state.get('question','')
    ans = state.get('answer','')
    decision:IsUse = isuse_chain.invoke({'question':qus,'answer':ans})

    return {
        'isuse': decision.isuse,
        'use_reason': decision.use_reason
    }

max_rewrite_tries = 2

def route_after_isuse(state:State) -> Literal['finalize','rewrite','no_answer_found']:
    if state.get('isuse') == 'useful' :
        return 'finalize'

    if state.get('isuse') == 'not_useful' and state.get('rewrite_tries',0) < max_rewrite_tries:
        return 'rewrite'
    
    return 'no_answer_found'

In [17]:
def accept_answer_node(state:State):
    return {}

In [18]:
def no_answer_found_node(state:State):
    return {'answer':'No answer found, because answer is not use full'}

In [19]:
def retrieve_node(state:State):
    q = state.get('retrieval_query') or state['question']
    return {'docs':retriever.invoke(q)}

In [20]:
class RewriteDecision(BaseModel):
    retrieval_query: str = Field(
        ...,
        description="Rewritten query optimized for vector retrieval against internal company PDFs."
        )
    

rewrite_prompt = ChatPromptTemplate.from_messages([
  (
            "system",
            "Rewrite the user's QUESTION into a query optimized for vector retrieval over INTERNAL company PDFs.\n\n"
            "Rules:\n"
            "- Keep it short (6–16 words).\n"
            "- Preserve key entities (e.g., NexaAI, plan names).\n"
            "- Add 2–5 high-signal keywords that likely appear in policy/pricing docs.\n"
            "- Remove filler words.\n"
            "- Do NOT answer the question.\n"
            "- Output JSON with key: retrieval_query\n\n"
            "Examples:\n"
            "Q: 'Do NexaAI plans include a free trial?'\n"
            "-> {{'retrieval_query': 'NexaAI free trial duration trial period plans'}}\n\n"
            "Q: 'What is NexaAI refund policy?'\n"
            "-> {{'retrieval_query': 'NexaAI refund policy cancellation refund timeline charges'}}"
        ),
        (
            "human",
            "QUESTION:\n{question}\n\n"
            "Previous retrieval query:\n{retrieval_query}\n\n"
            "Answer (if any):\n{answer}"
        ),
])

rewrite_chain = rewrite_prompt | llm.with_structured_output(RewriteDecision)

def rewrite_node(state:State):
    out:RewriteDecision = rewrite_chain.invoke({
        'question':state['question'],
        'retrieval_query': state.get('retrieval_query',''),
        'answer': state.get('answer','')
        
        })

    return {
        'retrieval_query':out.retrieval_query,
        'rewrite_tries': state.get('rewrite_tries',0) + 1
        
    }

In [21]:
def route_after_relevance(state:State) ->Literal['main_generation_node','no_relevant_docs']:
    if state.get('relevant_docs') and len(state.get('relevant_docs')) > 0:
        return "main_generation_node"
    return "no_relevant_docs"


In [22]:
def route_after_decide(state:State) -> Literal['generate_direct','retrieve']:
    if state['need_retrieval']:
        return 'retrieve'
    return 'generate_direct'


In [23]:
max_retries = 3

def route_after_issup(state:State) -> Literal['accept_answer','revise_answer','no_answer_found_node']:
    if state.get('issup') == 'fully_supported':
        return 'accept_answer'

    if state.get('retries',0) >= max_retries:
        return 'no_answer_found_node'
    return 'revise_answer'

In [24]:
g = StateGraph(State)

# --------------------
# Nodes
# --------------------
g.add_node("decide_retrieval", decide_retrieval_node)
g.add_node("generate_direct", generation_node)
g.add_node("retrieve", retrieve_node)
g.add_node('isrelevant',isrelevant_node)
g.add_node('main_generation_node',main_generation_node)
g.add_node("issup_node",issup_node)
g.add_node('revise_node',revise_node)
g.add_node('accept_answer_node',accept_answer_node)
g.add_node('isuse_node',isuse_node)
g.add_node('no_answer_found_node',no_answer_found_node)
g.add_node('no_relevant_docs', no_relevant_docs) 
g.add_node('rewrite_node',rewrite_node)





g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)
g.add_edge('retrieve','isrelevant')

g.add_conditional_edges(
    'isrelevant',
    route_after_relevance,
    {'no_relevant_docs':'no_relevant_docs','main_generation_node':'main_generation_node'}
    )

g.add_edge("main_generation_node",'issup_node') 

g.add_conditional_edges(
    'issup_node',
    route_after_issup,
    {'accept_answer':'accept_answer_node','revise_answer':'revise_node','no_answer_found_node':'no_answer_found_node'}
)

g.add_edge('revise_node','issup_node')
g.add_edge('accept_answer_node','isuse_node') 

g.add_conditional_edges(
    'isuse_node',
    route_after_isuse,
    {
        'finalize': END,
        'rewrite':'rewrite_node',
        'no_answer_found':'no_answer_found_node'
    }
    )
g.add_edge('rewrite_node','retrieve')
g.add_edge('no_answer_found_node',END)
g.add_edge('no_relevant_docs', END)



app = g.compile()


In [25]:
result = app.invoke(
    {
        "question": "computer dipertment ek ekjon principal teacher er nam bolo ",
        "need_retrieval": False,
        'relevant_docs':'',
        "docs": [],
        'retries':0,
        'issup': '',
        'evidence':'',
        "answer": "",
        'isuse': '',
        'use_reason':'',
        'retrieval_query': '',
        'rewrite_tries':0
    }
)

print('need retrieval: ',result['need_retrieval'])
print('len: ',len(result['docs']))
print('relevant docs: ',result['relevant_docs'])
print('relevant docs len: ',len(result['relevant_docs']))
print('issup: ',result['issup'])
print('retries: ',result['retries'])
print('evidence: ',result['evidence'])
print('isuse: ',result['isuse'])
print('rewrite retries: ',result['rewrite_tries'])
print('rewrite query: ',result['retrieval_query'])
print('isuse reason: ',result['use_reason'])
print('answer: ',result['answer'])







need retrieval:  True
len:  4
relevant docs:  [Document(id='cbfe3958-6c8f-47be-addf-f398fcc71706', metadata={'source': 'G:\\Genarative-AI\\Self Rag\\Document\\teachers_data.json', 'position': 'অধ্যক্ষ (অতিরিক্ত দায়িত্ব)', 'department': 'অতিরিক্ত দায়িত্ব', 'phone': '০১৭১১২৬২৩২৮', 'user_type': 'teacher', 'name': 'মোঃ ওমর ফারুক', 'seq_num': 1}, page_content='মোঃ ওমর ফারুক হলেন (চাঁপাইনবাবগঞ্জ পলিটেকনিক ইনস্টিটিউট / CNPI)-এর অধ্যক্ষ / principal (অতিরিক্ত দায়িত্ব)। তার পদবি হলো অধ্যক্ষ / Principal (অতিরিক্ত দায়িত্ব)। তিনি অতিরিক্ত দায়িত্ব বিভাগে কর্মরত আছেন। তার সাথে যোগাযোগ করতে পারবেন এই নম্বরে: ০১৭১১২৬২৩২৮। মোঃ ওমর ফারুক-এর ফোন নম্বর ০১৭১১২৬২৩২৮। যদি কেউ অতিরিক্ত দায়িত্ব বিভাগের শিক্ষকদের তথ্য জানতে চান, তাহলে মোঃ ওমর ফারুক (অধ্যক্ষ (অতিরিক্ত দায়িত্ব)) একজন গুরুত্বপূর্ণ যোগাযোগ ব্যক্তি।')]
relevant docs len:  1
issup:  fully_supported
retries:  0
evidence:  ['মোঃ ওমর ফারুক হলেন (চাঁপাইনবাবগঞ্জ পলিটেকনিক ইনস্টিটিউট / CNPI)-এর অধ্যক্ষ / principal (অতিরিক্ত দায়িত্ব)', 'মোঃ ওমর ফারুক-এর ফো

In [26]:
print('rewrite retries: ',result['relevant_docs'])

rewrite retries:  [Document(id='cbfe3958-6c8f-47be-addf-f398fcc71706', metadata={'source': 'G:\\Genarative-AI\\Self Rag\\Document\\teachers_data.json', 'position': 'অধ্যক্ষ (অতিরিক্ত দায়িত্ব)', 'department': 'অতিরিক্ত দায়িত্ব', 'phone': '০১৭১১২৬২৩২৮', 'user_type': 'teacher', 'name': 'মোঃ ওমর ফারুক', 'seq_num': 1}, page_content='মোঃ ওমর ফারুক হলেন (চাঁপাইনবাবগঞ্জ পলিটেকনিক ইনস্টিটিউট / CNPI)-এর অধ্যক্ষ / principal (অতিরিক্ত দায়িত্ব)। তার পদবি হলো অধ্যক্ষ / Principal (অতিরিক্ত দায়িত্ব)। তিনি অতিরিক্ত দায়িত্ব বিভাগে কর্মরত আছেন। তার সাথে যোগাযোগ করতে পারবেন এই নম্বরে: ০১৭১১২৬২৩২৮। মোঃ ওমর ফারুক-এর ফোন নম্বর ০১৭১১২৬২৩২৮। যদি কেউ অতিরিক্ত দায়িত্ব বিভাগের শিক্ষকদের তথ্য জানতে চান, তাহলে মোঃ ওমর ফারুক (অধ্যক্ষ (অতিরিক্ত দায়িত্ব)) একজন গুরুত্বপূর্ণ যোগাযোগ ব্যক্তি।')]


In [27]:
print('rewrite query: ',result['retrieval_query'])

rewrite query:  
